# Layers
Input → Embedding → LSTM → Dropout → Hidden Dense → Dropout → Output

# 1.Install Libraries

In [ ]:
!pip install datasets -q

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

# 2. Load Datasets

In [ ]:
dataset=load_dataset("cardiffnlp/tweet_eval","offensive")
train_data=dataset["train"]
val_data=dataset["validation"]
test_data=dataset["test"]
print(train_data[[0]])

In [ ]:
print(dataset)
print(train_data[0:2])

# 3. Convert dataset to lists

In [ ]:
print(train_data['label'])

In [ ]:
X_train=train_data['text']
y_train=np.array(train_data['label'])

X_val=val_data['text']
y_val=np.array(val_data['label'])

X_test=test_data['text']
y_test=np.array(test_data['label'])

# 4. Tokenize text

In [ ]:
lengths=[len(text.split()) for text in X_train]
print("Max length",np.max(lengths))
print("95th percentile:", np.percentile(lengths, 95))

In [ ]:
MAX_WORDS=10000
MAX_LEN=60

tokenizer=Tokenizer(num_words=MAX_WORDS,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq=tokenizer.texts_to_sequences(X_train)
X_val_seq=tokenizer.texts_to_sequences(X_val)
X_test_seq=tokenizer.texts_to_sequences(X_test)


# 5. Padding

In [ ]:
X_train_pad=pad_sequences(X_train_seq,maxlen=MAX_LEN,padding="post",truncating="post")
X_val_pad=pad_sequences(X_val_seq,maxlen=MAX_LEN,padding="post",truncating="post")
X_test_pad=pad_sequences(X_test_seq,maxlen=MAX_LEN,padding="post",truncating="post")


# 6. Build LSTM model

In [ ]:
model=Sequential([
    Embedding(input_dim=MAX_WORDS,output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(32,activation="relu"),
    Dropout(0.3),
    Dense(1,activation="sigmoid")
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

# 7. Train model

In [ ]:
history=model.fit(X_train_pad,
                  y_train,
                  validation_data=(X_val_pad,y_val),
                  epochs=5,
                  batch_size=32,
                 )

# 8. Prediction & Error analysis

In [ ]:
y_prob=model.predict(X_test_pad).ravel()#flatten the predicted output
y_pred=(y_prob>=0.5).astype(int)
print(classification_report(y_test,y_pred, target_names=["non-offensive", "offensive"]))
print(confusion_matrix(y_test,y_pred))

The model achieves reasonable performance, but still struggles with the offensive class.

# 9. Add neutral class

In [ ]:
def predict_with_uncertainty(prob):
    if(prob>=0.6):
        return "offensive"
    elif prob<=0.4:
        return "non-offensive"
    else:
        return "neutral/uncertain"

In [ ]:
uncertain_preds=[predict_with_uncertainty(p) for p in y_prob]
result_df=pd.DataFrame({
    "text":X_test,
    "true_label":y_test,
    "toxic_probability":y_prob,
    "prediction": uncertain_preds
})
result_df.head(20)

# Transformer

# 10. install libraries

In [36]:
!pip install transformers evaluate accelerate -q

# 11. Load dataset

In [37]:
from datasets import load_dataset
dataset2=load_dataset("cardiffnlp/tweet_eval","offensive")

train_dataset2=dataset2["train"]
val_dataset2=dataset2["validation"]
test_dataset2=dataset2["test"]

print(dataset2)
print(train_dataset2[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 11916
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 860
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1324
    })
})
{'text': '@user Bono... who cares. Soon people will understand that they gain nothing from following a phony celebrity. Become a Leader of your people instead or help and support your fellow countrymen.', 'label': 0}


# 12. Tokenize for Transformer

In [38]:
from transformers import AutoTokenizer

MODEL_NAME="distilbert-base-uncased"
tokenizer2=AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer2(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )
tokenized_dataset=dataset2.map(tokenize_function,batched=True)

print(tokenized_dataset)
print(tokenized_dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 11916
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 860
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1324
    })
})
{'text': '@user Bono... who cares. Soon people will understand that they gain nothing from following a phony celebrity. Become a Leader of your people instead or help and support your fellow countrymen.', 'label': 0, 'input_ids': [101, 1030, 5310, 23648, 1012, 1012, 1012, 2040, 14977, 1012, 2574, 2111, 2097, 3305, 2008, 2027, 5114, 2498, 2013, 2206, 1037, 6887, 16585, 8958, 1012, 2468, 1037, 3003, 1997, 2115, 2111, 2612, 2030, 2393, 1998, 2490, 2115, 3507, 2406, 3549, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids':

In [39]:
print(tokenized_dataset.column_names)

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'validation': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}


# 13. Prepare labels

In [40]:
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","labels"]
)

# 14. Build Transformer model

In [41]:
from transformers import AutoModelForSequenceClassification

model=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 15. Metrics

In [42]:
import numpy as np
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
def compute_metrics(eval_pred):
    logits,labels=eval_pred
    preds=np.argmax(logits,axis=1)
    precision,recall,f1,_=precision_recall_fscore_support(
        labels,
        preds,
        average="macro"
    )
    acc=accuracy_score(labels,preds)

    return{
        "accuracy":acc,
        "macro_precision":precision,
        "macro_recall":recall,
        "macro_f1":f1
    }

# 16. Train Transformer

In [45]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,No log,0.900989,0.792296,0.773490,0.791957,0.779152
2,0.890775,0.883557,0.796073,0.774686,0.780531,0.777343
3,0.699996,0.908857,0.801360,0.781737,0.775374,0.778300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1119, training_loss=0.7740365131504308, metrics={'train_runtime': 170.722, 'train_samples_per_second': 209.393, 'train_steps_per_second': 6.555, 'total_flos': 591930570894336.0, 'train_loss': 0.7740365131504308, 'epoch': 3.0})

# 17. Evaluate on test set

In [47]:
test_result=trainer.evaluate(tokenized_dataset["test"])
print(test_result)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.8171682953834534, 'eval_accuracy': 0.8081395348837209, 'eval_macro_precision': 0.7629874336053892, 'eval_macro_recall': 0.786491935483871, 'eval_macro_f1': 0.7723477063911095, 'eval_runtime': 1.4732, 'eval_samples_per_second': 583.777, 'eval_steps_per_second': 18.328, 'epoch': 3.0}


In [49]:
from sklearn.metrics import classification_report,confusion_matrix

pred_output=trainer.predict(tokenized_dataset["test"])
logits=pred_output.predictions
y_pred=np.argmax(logits,axis=1)
y_true=pred_output.label_ids
print(classification_report(
    y_true,
    y_pred,
    target_names=["non-offensive","offensive"]
))

print(confusion_matrix(y_true,y_pred))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


               precision    recall  f1-score   support

non-offensive       0.89      0.84      0.86       620
    offensive       0.63      0.74      0.68       240

     accuracy                           0.81       860
    macro avg       0.76      0.79      0.77       860
 weighted avg       0.82      0.81      0.81       860

[[518 102]
 [ 63 177]]


# 18. Save Transformer model

In [51]:
trainer.save_model("./saved_transformer")
tokenizer2.save_pretrained("./saved_transformer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_transformer/tokenizer_config.json',
 './saved_transformer/tokenizer.json')

In [52]:
!zip -r saved_transformer.zip ./saved_transformer

  adding: saved_transformer/ (stored 0%)
  adding: saved_transformer/tokenizer_config.json (deflated 42%)
  adding: saved_transformer/training_args.bin (deflated 53%)
  adding: saved_transformer/model.safetensors (deflated 8%)
  adding: saved_transformer/tokenizer.json (deflated 71%)
  adding: saved_transformer/config.json (deflated 49%)


In [53]:
from IPython.display import FileLink

FileLink("/kaggle/working/saved_transformer.zip")

/kaggle/working/saved_transformer.zip